# Config

## Add project root to Python path inside the notebook

In [1]:
import sys, os

# Go one level up from the notebook folder to project root
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added:", project_root)

Project root added: c:\Users\atulsehgal\OneDrive\Documents\repos\talk-to-my-data-semantic


# Validation

## Test- Semantic model loads correctly

In [2]:
from src.semantic.semantic_model import SemanticModel

model = SemanticModel.from_yaml(
    os.path.join(project_root, "src/semantic/model_tpch.yml")
)

print(model.tables.keys())
print(model.measures.keys())
print(model.dimensions.keys())
print(model.relationships[0])

dict_keys(['customer', 'orders', 'lineitem', 'part', 'partsupp', 'supplier', 'nation', 'region'])
dict_keys(['revenue', 'gross_sales', 'order_count', 'avg_discount', 'quantity_sold'])
dict_keys(['order_date', 'ship_date', 'customer_name', 'customer_segment', 'region_name', 'nation_name', 'supplier_name', 'part_name', 'product_brand'])
Relationship(from_table='orders', from_column='o_custkey', to_table='customer', to_column='c_custkey', type='many_to_one', role='customer_hierarchy', description='Each order belongs to a single customer.')


## Test- Semantic resolver

In [3]:
from utils.config_loader import load_env
load_env()

✅ Loaded environment variables from configs/dev.env


In [4]:
from src.semantic.semantic_resolver import SemanticResolver
resolver = SemanticResolver(model)

In [5]:
from dataclasses import asdict
import json

Evaluate each one for:

✔ Correct measure selection

✔ Correct dimension selection

✔ Correct time filter

✔ Correct grouping

✔ Correct semantic matching

✔ Whether the LLM respected your model’s rules

✔ Whether join-related fields were interpreted correctly

✔ Any inconsistencies or potential improvements

=============== EXAMPLE 1 =================

In [6]:
plan = resolver.plan_from_question("sales in last 3 months")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "revenue",
    "expression": "SUM(l_extendedprice * (1 - l_discount))",
    "table": "lineitem",
    "description": "Net sales revenue after discount at line level.",
    "synonyms": [
      "sales",
      "net_sales",
      "turnover",
      "sales_revenue"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= DATEADD(MONTH, -3, CURRENT_DATE)",
  "group_by_dimensions": []
}


⭐ Final Verdict

✔ Correct measure

✔ Correct time dimension

✔ Correct time filter

✔ Correct grouping (none)

✔ No errors or hallucinations

❗ SQL column names will be corrected in Step 5

This is a solid semantic reasoning output.

=============== EXAMPLE 2 =================

In [7]:
plan = resolver.plan_from_question("top 5 customers by revenue")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "revenue",
    "expression": "SUM(l_extendedprice * (1 - l_discount))",
    "table": "lineitem",
    "description": "Net sales revenue after discount at line level.",
    "synonyms": [
      "sales",
      "net_sales",
      "turnover",
      "sales_revenue"
    ]
  },
  "time_dimension": null,
  "time_filter": null,
  "group_by_dimensions": [
    {
      "name": "customer_name",
      "table": "customer",
      "column": "c_name",
      "description": "Customer name.",
      "synonyms": [
        "customer",
        "client"
      ],
      "time_grains": []
    }
  ]
}


⭐ Final Verdict

**This output is 95% perfect.

VERY GOOD semantic reasoning.**

✔ Correct measure

✔ Correct grouping

✔ No unnecessary time dimension

✔ No unnecessary time filter

❗ Minor improvement needed for LIMIT 5 (handled later)

=============== EXAMPLE 3 =================

In [8]:
plan = resolver.plan_from_question("revenue by month this year")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "revenue",
    "expression": "SUM(l_extendedprice * (1 - l_discount))",
    "table": "lineitem",
    "description": "Net sales revenue after discount at line level.",
    "synonyms": [
      "sales",
      "net_sales",
      "turnover",
      "sales_revenue"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= DATE_TRUNC('month', CURRENT_DATE) AND order_date < DATEADD('month', 1, DATE_TRUNC('month', CURRENT_DATE))",
  "group_by_dimensions": [
    {
      "name": "order_date",
      "table": "orders",
      "column": "o_orderdate",
      "description": "Order entry date.",
      "synonyms": [
        "date",
        "transaction_date",
        "order_day"
     

⭐ Final Verdict

✔ Correct measure

✔ Correct time dimension

✔ Correct MTD time filter (perfect SQL logic)

✔ Correct grouping

✔ No hallucinations

✔ No unnecessary joins or dims

❗ Column name translation deferred to SQL Generator

❗ Grain not explicit yet (will be added later)

This is a very strong semantic interpretation (95%+ correct).

=============== EXAMPLE 4 =================

In [9]:
plan = resolver.plan_from_question("units sold by brand last quarter")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "quantity_sold",
    "expression": "SUM(l_quantity)",
    "table": "lineitem",
    "description": "Total quantity sold.",
    "synonyms": [
      "volume",
      "units_sold"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= DATE_TRUNC('quarter', CURRENT_DATE) - INTERVAL '1 quarter' AND order_date < DATE_TRUNC('quarter', CURRENT_DATE)",
  "group_by_dimensions": [
    {
      "name": "product_brand",
      "table": "part",
      "column": "p_brand",
      "description": "Product brand",
      "synonyms": [
        "brand",
        "product_brand",
        "brand_name"
      ],
      "time_grains": []
    }
  ]
}


⭐ Final Verdict

✔ Correct measure

✔ Correct time dimension

✔ Correct quarterly time filter

✔ Correct grouping (brand)

✔ Correct join inference

✔ Perfect synonym recognition

✔ No hallucinations

❗ Minor: SQL column-name translation will be handled in SQL generator

=============== EXAMPLE 5 =================

In [10]:
plan = resolver.plan_from_question("number of orders placed in the last 10 days")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "order_count",
    "expression": "COUNT(DISTINCT o_orderkey)",
    "table": "orders",
    "description": "Number of distinct orders.",
    "synonyms": [
      "transactions",
      "num_orders",
      "order_volume"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= CURRENT_DATE - INTERVAL '10 days'",
  "group_by_dimensions": []
}


⭐ Overall Verdict (same structured style)

✔ Correct measure

✔ Correct time dimension

✔ Correct 10-day rolling filter

✔ Correct grouping (none)

✔ Correct table-level interpretation

✔ No hallucinations

✔ No incorrect joins

❗ Column-name translation for SQL handled later

⭐ Overall: 100% correct

## Test- Join Graph

In [11]:
from src.semantic.join_graph import JoinGraph

In [12]:
jg = JoinGraph.from_model(model)

jg.print_graph()

orders:
   orders.o_custkey  →  customer.c_custkey
   orders.o_orderkey  →  lineitem.l_orderkey

customer:
   customer.c_nationkey  →  nation.n_nationkey

lineitem:
   lineitem.l_orderkey  →  orders.o_orderkey
   lineitem.l_partkey  →  part.p_partkey
   lineitem.l_suppkey  →  supplier.s_suppkey

nation:
   nation.n_regionkey  →  region.r_regionkey

supplier:
   supplier.s_nationkey  →  nation.n_nationkey

region:

part:

partsupp:
   partsupp.ps_partkey  →  part.p_partkey
   partsupp.ps_suppkey  →  supplier.s_suppkey



## Test- Relationship objects are loading role correctly

In [13]:
import importlib

import src.semantic.join_graph as join_graph_module
import src.semantic.join_resolver as join_resolver_module

importlib.reload(join_graph_module)
importlib.reload(join_resolver_module)

<module 'src.semantic.join_resolver' from 'c:\\Users\\atulsehgal\\OneDrive\\Documents\\repos\\talk-to-my-data-semantic\\src\\semantic\\join_resolver.py'>

### Confirm that your Relationship objects are loading role correctly

In [14]:
for rel in model.relationships:
    print(rel.from_table, rel.from_column, "→", rel.to_table, rel.to_column, "| role:", rel.role)

orders o_custkey → customer c_custkey | role: customer_hierarchy
lineitem l_orderkey → orders o_orderkey | role: order
orders o_orderkey → lineitem l_orderkey | role: order
customer c_nationkey → nation n_nationkey | role: customer_hierarchy
supplier s_nationkey → nation n_nationkey | role: supplier_hierarchy
nation n_regionkey → region r_regionkey | role: customer_hierarchy
lineitem l_partkey → part p_partkey | role: product_hierarchy
lineitem l_suppkey → supplier s_suppkey | role: supplier_hierarchy
partsupp ps_partkey → part p_partkey | role: product_hierarchy
partsupp ps_suppkey → supplier s_suppkey | role: supplier_hierarchy


### Confirm that roles are being put into JoinEdges

In [15]:
for src, edges in jg.graph.items():
    for e in edges:
        if e.target == "region":
            print(e)

JoinEdge(source='nation', target='region', source_column='n_regionkey', target_column='r_regionkey', role='customer_hierarchy', description='Each nation is associated with a region.')


### Confirm that find_path() is receiving preferred roles

In [16]:
path = jg.find_path(
    start="lineitem",
    target="region",
    preferred_roles={"customer_hierarchy"}
)

for e in path:
    print(e.source, "→", e.target, "| role:", e.role)


lineitem → orders | role: order
orders → customer | role: customer_hierarchy
customer → nation | role: customer_hierarchy
nation → region | role: customer_hierarchy


## Test- Join Path Resolution

In [17]:
jg.show_path_raw("lineitem", "region")
jg.show_path_semantic("lineitem", "region", preferred_roles={"customer_hierarchy"})

jg.show_path_raw("orders", "supplier")
jg.show_path_semantic("orders", "supplier", preferred_roles={"supplier_hierarchy"})

jg.show_path_raw("lineitem", "part")
jg.show_path_semantic("lineitem", "part")

jg.show_path_raw("customer", "region")
jg.show_path_semantic("customer", "region", preferred_roles={"customer_hierarchy"})


RAW shortest path from 'lineitem' to 'region':
   lineitem.l_suppkey → supplier.s_suppkey | role: supplier_hierarchy
   supplier.s_nationkey → nation.n_nationkey | role: supplier_hierarchy
   nation.n_regionkey → region.r_regionkey | role: customer_hierarchy

SEMANTIC path from 'lineitem' to 'region' (roles={'customer_hierarchy'}):
   lineitem.l_orderkey → orders.o_orderkey | role: order
   orders.o_custkey → customer.c_custkey | role: customer_hierarchy
   customer.c_nationkey → nation.n_nationkey | role: customer_hierarchy
   nation.n_regionkey → region.r_regionkey | role: customer_hierarchy

RAW shortest path from 'orders' to 'supplier':
   orders.o_orderkey → lineitem.l_orderkey | role: order
   lineitem.l_suppkey → supplier.s_suppkey | role: supplier_hierarchy

SEMANTIC path from 'orders' to 'supplier' (roles={'supplier_hierarchy'}):
   orders.o_orderkey → lineitem.l_orderkey | role: order
   lineitem.l_suppkey → supplier.s_suppkey | role: supplier_hierarchy

RAW shortest path fr

Correct
This is the supplier geography path.

IMPORTANT NOTE:
There are two valid geography paths in TPCH:

Path A (via customer):
lineitem → orders → customer → nation → region

Path B (via supplier):
lineitem → supplier → nation → region


Both are correct in TPCH — depending on whether you mean:

customer’s region

supplier’s region

Right now, BFS is picking the shortest path, which is supplier region (3 hops) instead of customer region (4 hops).

➡️ This is correct per BFS definition.
➡️ Later, we will allow the resolver to choose the correct semantic path based on dimension context (e.g., region of customer vs region of supplier).

For now, your BFS is functioning perfectly.

In [18]:
jg.show_path_raw("orders", "supplier")
jg.show_path_raw("lineitem", "part")
jg.show_path_raw("customer", "region")


RAW shortest path from 'orders' to 'supplier':
   orders.o_orderkey → lineitem.l_orderkey | role: order
   lineitem.l_suppkey → supplier.s_suppkey | role: supplier_hierarchy

RAW shortest path from 'lineitem' to 'part':
   lineitem.l_partkey → part.p_partkey | role: product_hierarchy

RAW shortest path from 'customer' to 'region':
   customer.c_nationkey → nation.n_nationkey | role: customer_hierarchy
   nation.n_regionkey → region.r_regionkey | role: customer_hierarchy


## Test- Build Join Path Resolver for SemanticPlan

In [19]:
from src.semantic.join_resolver import JoinResolver
from src.semantic.semantic_resolver import SemanticResolver

In [20]:
joiner = JoinResolver(jg)

In [21]:
plan = resolver.plan_from_question("revenue by region last year")
print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "revenue",
    "expression": "SUM(l_extendedprice * (1 - l_discount))",
    "table": "lineitem",
    "description": "Net sales revenue after discount at line level.",
    "synonyms": [
      "sales",
      "net_sales",
      "turnover",
      "sales_revenue"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= DATE_TRUNC('YEAR', CURRENT_DATE) - INTERVAL '1 YEAR' AND order_date < DATE_TRUNC('YEAR', CURRENT_DATE)",
  "group_by_dimensions": [
    {
      "name": "region_name",
      "table": "region",
      "column": "r_name",
      "description": "Region name.",
      "synonyms": [
        "region",
        "geographic_region"
      ],
      "time_grains": []
  

In [22]:
edges = joiner.joins_for_plan(plan)

for edge in edges:
    print(edge)

print("\n")

for e in edges:
    print(f"{e.source}.{e.source_column} → {e.target}.{e.target_column}")

JoinEdge(source='lineitem', target='orders', source_column='l_orderkey', target_column='o_orderkey', role='order', description='Each line item belongs to a single order.')
JoinEdge(source='orders', target='customer', source_column='o_custkey', target_column='c_custkey', role='customer_hierarchy', description='Each order belongs to a single customer.')
JoinEdge(source='customer', target='nation', source_column='c_nationkey', target_column='n_nationkey', role='customer_hierarchy', description='Each customer is associated with a nation.')
JoinEdge(source='nation', target='region', source_column='n_regionkey', target_column='r_regionkey', role='customer_hierarchy', description='Each nation is associated with a region.')


lineitem.l_orderkey → orders.o_orderkey
orders.o_custkey → customer.c_custkey
customer.c_nationkey → nation.n_nationkey
nation.n_regionkey → region.r_regionkey


## Test- SQL Generator

In [23]:
from src.semantic.sql_generator import SQLGenerator

In [24]:
sqlgen = SQLGenerator()

In [25]:
question = "revenue by region last year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)

SELECT
  SUM(l_extendedprice * (1 - l_discount)) AS revenue,
  region.r_name AS region_name
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
JOIN customer ON orders.o_custkey = customer.c_custkey
JOIN nation ON customer.c_nationkey = nation.n_nationkey
JOIN region ON nation.n_regionkey = region.r_regionkey
WHERE orders.o_orderdate >= DATE_TRUNC('YEAR', DATEADD('YEAR', -1, CURRENT_DATE)) AND orders.o_orderdate < DATE_TRUNC('YEAR', CURRENT_DATE)
GROUP BY region.r_name


In [26]:
import inspect
from src.semantic.join_graph import JoinGraph

print(inspect.getsource(JoinGraph.find_path))

    def find_path(
        self,
        start: str,
        target: str,
        preferred_roles: Optional[Set[str]] = None,
    ) -> Optional[List[JoinEdge]]:
        """
        Role-aware join path finder.

        What this method does:
        ------------------------------------
        We want to find a join path between two tables (start → target).
        But not all join paths are equally meaningful semantically.

        Example:
            lineitem → supplier → nation → region      (shorter, but supplier region)
            lineitem → orders → customer → nation → region  (longer, but customer region)

        Business meaning prefers customer geography, not supplier geography.

        Therefore, we use TWO ranking criteria:

        1. PRIMARY RANK: number of edges whose role is in `preferred_roles`
           (e.g., {"customer_hierarchy"}). More matches = better.

        2. SECONDARY RANK: number of hops (path length).
           Among paths with the same role score, c

In [27]:
for src, edges in jg.graph.items():
    for e in edges:
        print(e)

JoinEdge(source='orders', target='customer', source_column='o_custkey', target_column='c_custkey', role='customer_hierarchy', description='Each order belongs to a single customer.')
JoinEdge(source='orders', target='lineitem', source_column='o_orderkey', target_column='l_orderkey', role='order', description='Orders have multiple line items.')
JoinEdge(source='customer', target='nation', source_column='c_nationkey', target_column='n_nationkey', role='customer_hierarchy', description='Each customer is associated with a nation.')
JoinEdge(source='lineitem', target='orders', source_column='l_orderkey', target_column='o_orderkey', role='order', description='Each line item belongs to a single order.')
JoinEdge(source='lineitem', target='part', source_column='l_partkey', target_column='p_partkey', role='product_hierarchy', description='Line items reference a specific part.')
JoinEdge(source='lineitem', target='supplier', source_column='l_suppkey', target_column='s_suppkey', role='supplier_hie